<a href="https://colab.research.google.com/github/Samvite1990/Machine-learning/blob/main/%D0%9B%D0%A01_%D0%A1%D0%B0%D0%BC%D1%81%D0%BE%D0%BD%D0%BE%D0%B2_%D0%92%D0%B0%D1%80%D0%B8%D0%B0%D0%BD%D1%825.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Цель работы:
Изучение алгоритмов и методов кластеризации на практике.

#Описание датасета по заданию:
Файл "5gym_churn.csv" - информация о клиентах фитнес-клуба.

По описанию из задания датасет содержит следующие данные:

- Пол
- Близкое расположение
- Сотрудник компании партнера
- По промо друзей
- Указан ли телефон
- Длительность текущего абонемента
- Посещение групповых занятий
- Возраст
- Средние траты на дополнительные услуги
- Количество месяцев до окончания абонемента
- Время с момента первого обращения в фитнес-центр
- Средняя частота посещений в неделю за все время
- Средняя частота посещений в неделю за предыдущий месяц

Целевой переменной выбрана: **Churn** - факт ухода из клуба

## 1. Загрузка датасета

Для начала необходимо импортировать основные библиотеки для работы с данными, визуализации и машинного обучения.

Необходимо загрузить следующие модули:

* **pandas (pd)** - для загрузки, хранения и первичной обработки табличных данных (функции read_csv, head, info, describe, isnull, duplicated);
* **numpy (np)** - для числовых операций;
* **matplotlib.pyplot (plt)** - для построения графиков;
* **seaborn (sns)** - для статистической визуализации (функция pairplot);

После импорта библиотек датасет был загружен с помощью функции pd.read_csv(). Теперь нужно оценить структуру, размер, типы данных, первые строки и базовые статистики датасета.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Загрузка данных
df = pd.read_csv('5gym_churn.csv')

# Первичный просмотр данных
print("Размер датасета:", df.shape)
print("\nПервые 5 строк:")
display(df.head())

print("\nОбщая информация о типах данных:")
df.info()

Размер датасета: (4000, 14)

Первые 5 строк:


,gender,Near_Location,Partner,Promo_friends,Phone,Contract_period,Group_visits,Age,Avg_additional_charges_total,Month_to_end_contract,Lifetime,Avg_class_frequency_total,Avg_class_frequency_current_month,Churn
0,1,1,1,1,0,6,1,29,14.227470,5.0,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12.0,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1.0,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12.0,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1.0,3,1.113884,1.120078,0



Общая информация о типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   gender                             4000 non-null   int64  
 1   Near_Location                      4000 non-null   int64  
 2   Partner                            4000 non-null   int64  
 3   Promo_friends                      4000 non-null   int64  
 4   Phone                              4000 non-null   int64  
 5   Contract_period                    4000 non-null   int64  
 6   Group_visits                       4000 non-null   int64  
 7   Age                                4000 non-null   int64  
 8   Avg_additional_charges_total       4000 non-null   float64
 9   Month_to_end_contract              4000 non-null   float64
 10  Lifetime                           4000 non-null   int64  
 11  Avg_class_frequency_to

После исполнения кода видно, что датасет содержит 14 столбцов и 4 тысячи строк.

Все признаки уже представлены в числовом формате.

Бинарные признаки (пол, близкое расположение, сотрудник компании-партнёра и т.д.) закодированы как 0 и 1, количественные признаки, такие как возраст, средние траты, длительность абонемента имеют тип float64 или int64.

Целевой переменной является Churn (1 - ушёл, 0 - остался).


##2. Предварительная обработка данных

Перед построением модели важно убедиться в отсутствии пропущенных значений и дубликатов, которые могут исказить результаты кластеризации.

Для проверки данных необходимо использховать методы:
* **isnull().sum()** - подсчёт пропусков по каждому столбцу;
* **duplicated().sum()** - подсчёт дублирующихся строк;
* **fillna()** - заполнение пропусков (если они есть) медианой для числовых признаков;
* **drop_duplicates()** - удаление дубликатов.

Поскольку в датасете все признаки уже числовые, дополнительное кодирование категориальных переменных не требуется.

In [ ]:
# Проверка на пропуски
print("Количество пропусков:")
print(df.isnull().sum())

# Проверка на дубликаты
print(f"\nКоличество дубликатов: {df.duplicated().sum()}")

# Заполнение пропусков
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Удаление дубликатов
df = df.drop_duplicates()

print(f"\nРазмер датасета после обработки: {df.shape}")

Анализ выполнения кода показал, что в наборе данных нет пропущенных значений и дубликатов строк.

Все признаки являются количественными, поэтому предварительная обработка завершена и данные полностью готовы к дальнейшей работе.


## 3. Выделение целевой переменной

Для дальнейшей работы необходимо выделить целевую переменную из данных датасета.

Данные необходимо кластерезировать и выделить ключевые признаки для дальнейшего обучения модели без учителя.

In [ ]:
# Выделение целевой переменной
target = df['Churn']

# Формирование матрицы признаков (все столбцы, кроме Churn)
X = df.drop('Churn', axis=1)

print("Размер матрицы признаков X:", X.shape)
print("Размер целевой переменной:", target.shape)

print("\nРаспределение целевой переменной:")
print(target.value_counts())
print(f"\nДоля ушедших клиентов: {target.mean()*100:.2f}%")
print(f"\nДоля оставшихся клиентов: {100 - target.mean()*100:.2f}%")

После разделения данных получилась матрица признаков размером (4000, 13) и вектор целевой переменной target.

Распределение классов показывает, что 73.47 процентов клиентов остались в клубе, а меньшая часть (26.52%) ушли.

В дальнейшем матрицe нужно будет использовать для кластеризации, а target - для анализа результатов.


## 4. Составление матрицы диаграмм рассеяния

Для визуального анализа взаимосвязей между признаками и их влияния на целевую переменную нужно построить матрицу диаграмм рассеяния с помощью функции sns.pairplot() из библиотеки seaborn.

Чтобы график не был перегружен, необходимо выбрать несколько наиболее информативных числовых признаков, например: Age, Avg_additional_charges_total, Lifetime и Avg_class_frequency_total.

Построение графика поможет оценить различаются ли клиенты с Churn=0 и Churn=1 по этим признакам, и есть ли потенциальные кластеры.

In [ ]:
# Выбираем ключевые признаки для визуализации
cols_to_plot = ['Age', 'Avg_additional_charges_total', 'Lifetime', 'Avg_class_frequency_total', 'Churn']

plt.figure(figsize=(10, 8))
sns.pairplot(df[cols_to_plot], hue='Churn', palette='Set1', diag_kind='kde', plot_kws={'alpha': 0.6})
plt.suptitle('Матрица диаграмм рассеяния (целевая переменная Churn)', y=1.02)
plt.show()

На построенной матрице диаграмм рассеяния хорошо видны различия между клиентами, которые остались в клубе (красный цвет), и теми, кто ушёл (синий цвет).

По графикам распределения видно, что клиенты, которые ушли из клуба, в среднем моложе, а пик их распределения приходится на 25–27 лет, тогда как оставшиеся клиенты имеют более равномерное распределение с пиком около 30 лет и охватывают более широкий возрастной диапазон (до 40 лет).

Средние траты на дополнительные услуги (Avg_additional_charges_total) у ушедших клиентов заметно ниже и распределение сконцентрировано в области 100-150, тогда как у тех кт о остаётся пик приходится на 200-250. Это говорит о том, что клиенты, которые активно пользуются дополнительными услугами, реже покидают клуб.

Время с момента первого посещения (Lifetime) у ушедших клиентов сконцентрировано в области 0-5 месяцев, то есть они уходят практически сразу после прихода. У оставшихся клиентов распределение имеет длинный правый хвост, многие являются клиентами клуба более 10-20 месяцев.

Средняя частота посещений (Avg_class_frequency_total) у ушедших клиентов в районе 1-2 посещений в неделю, причём значительная часть имеет частоту, близкую к нулю. Лояльные клиенты посещают клуб в среднем 2-3 раза в неделю.

По scatter-графикам видно наиболее чёткое разделение классов там, где участвует признак Lifetime. Синие точки плотно сгруппированы в левом нижнем углу графиков. Показатели Lifetime / Avg_class_frequency_total и Lifetime / Avg_additional_charges_total показывают, что это клиенты с коротким стажем, низкой частотой посещений и небольшими тратами. Красные точки распределены значительно равномернее и занимают практически всю площадь графиков.

На графике Age / Lifetime также хорошо видно, что ушедшие клиенты - это преимущественно молодые люди с коротким стажем посещения клуба.

Из визуального анализа можно сделать вывод о том, что данные имеют выраженную кластерную структуру. Признак Lifetime является наиболее информативным для разделения клиентов на группы.
Алгоритм кластеризации, скорее всего, выделит как минимум два кластера, соответствующих лояльным клиентам и клиентам с высоким риском ухода.


## 5. Стандартизация числовых данных

Методы кластеризации, основанные на вычислении расстояний, такие как "K-means", чувствительны к масштабу признаков.

Для приведения данных к единому масштабу нужно использовать стандартизацию с помощью класса StandardScaler из модуля sklearn.preprocessing.
Также в работе используется метод **fit_transform()** для обучения стандартизатора на данных и их преобразования.

После обработки данные должны ринять вид стандартного нормального распределения, что минимизирует влияние признаков с большими числовыми диапазонами.

Ожидается, что после стандартизации все признаки будут иметь одинаковый масштаб.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Матрица признаков после стандартизации (первые 5 строк):")
print(X_scaled[:5])

print(f"\nСреднее по признакам: {X_scaled.mean(axis=0).round(6)}")
print(f"Стандартное отклонение по признакам: {X_scaled.std(axis=0).round(6)}")

Анализ результатов выполнения кода показал, что стандартизация прошла корректно.

Матрица признаков X_scaled содержит преобразованные значения которые могут быть как положительными, так и отрицательными, что соответствует отклонениям от среднего значения.

Статистики показывают, что среднее значение по каждому из 13 признаков стало равно 0, а стандартное отклонение по каждому признаку стало равно 1. Это означает, что все признаки теперь находятся в одной размерности и ни один из них не будет иметь преимущества при вычислении евклидовых расстояний между объектами в алгоритме "K-means".


# 6. Кластеризация методом "K-means" и подбор оптимального количества кластеров

Так-как в датасете нет чёткого разделения клиентов по группам, то необходимо выполнить кластеризацию.

Для кластеризации используется класс KMeans из модуля sklearn.cluster.

Основные параметры:
* **n_clusters** - число кластеров;
* **random_state** - фиксация случайного состояния для воспроизводимости результата;
* **n_init** - количество запусков алгоритма с разными центроидами.

Для выбора оптимального числа кластеров k будет применен "метод локтя" и коэффициент силуэта.

Атрибут inertia_ модели, показывает сумму квадратов расстояний от объектов до центроидов их кластеров.

Коэффициент силуэта вычисляется с помощью функции silhouette_score() из модуля sklearn.metrics и показывает, насколько объект похож на объекты своего кластера по сравнению с объектами других кластеров

Переберём значения k от 2 до 9, обучим модель для каждого значения и построим совмещённый график инерции и силуэта.

Задачей данного этапа является нахождение параметра k, при котором инерция перестаёт значительно снижаться, а силуэт достигнет максимума.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertia = []
silhouette_scores = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, kmeans.labels_))

# Визуализация результатов
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:blue'
ax1.set_xlabel('Количество кластеров (k)')
ax1.set_ylabel('Инерция (Within-Cluster Sum of Square)', color=color)
ax1.plot(k_range, inertia, marker='o', color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Коэффициент силуэта (silhouette scores) ', color=color)
ax2.plot(k_range, silhouette_scores, marker='s', color=color)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Метод локтя и коэффициент силуэта для подбора k')
plt.grid(True, alpha=0.3)
plt.show()

Анализ графика показывает, что выраженного "локтя" на кривой инерции не наблюдается. График плавно снижается от k = 2 до k = 9. Это означает, что данные не имеют ярко выраженной кластерной структуры с чёткими границами между группами.

Коэффициент силуэта достигает своего максимума при k = 2, после чего снижается и держится в диапазоне 0.12 - 0.14.

Так - как максимальный коэффициент силуэта достигается при k = 2, а целевая переменная имеет бинарные значения, то для дальнейшей работы можно выделить 2 группы.

In [ ]:
# Финальное обучение модели k-means с k=2
kmeans_final = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans_final.fit_predict(X_scaled)

print(f"Модель обучена. Количество кластеров: 2")
print(f"Распределение объектов по кластерам:")
print(pd.Series(clusters).value_counts().sort_index())
print(f"\nИнерция модели: {kmeans_final.inertia_:.0f}")
print(f"Silhouette Score: {silhouette_score(X_scaled, clusters):.4f}")

Видно, что алгоритм распределил данные на два кластера.
Коэффициент силуэта равен 0.1647 и близок к нулю, что может говорить о том, что кластеры перекрывают друг друга и не имеют выраженных границ.


## 7. Анализ кластеров

Далее нужно добавить метки кластеров обратно в копию исходного датафрейма и с помощью метода groupby() библиотеки pandas сгруппировать данные по номеру кластера и вычислить средние значения для всех числовых признаков, включая целевую переменную Churn.

Это поможет увидеть четкие различия в поведении клиентов между кластерами.

In [ ]:
# Добавляем метки кластеров в копию исходного датафрейма
df_analysis = df.copy()
df_analysis['Cluster'] = clusters

# Вычисляем средние показатели по каждому признаку для каждого кластера
cluster_summary = df_analysis.groupby('Cluster').mean().round(2)

print("Средние показатели по признакам для каждого кластера:")
display(cluster_summary)


Из результатов работы кода видно, что кластер 0 объединяет клиентов с высоким риском ухода - это новички с коротким абонементом, низкой активностью и небольшими тратами, а кластер 1 - это лояльные клиенты с длинным абонементом, высокой частотой посещений.

**В кластере 0 сосредоточились клиенты с высоким риском оттока.**

Доля ушедших клиентов в этом кластере составляет 40%, что значительно выше среднего по выборке. Эти клиенты характеризуются коротким абонементом (Contract_period = 1.72 месяца), небольшим временем с момента первого обращения (Lifetime = 3.11 месяца) и абонементом, который скоро заканчивается (Month_to_end_contract = 1.65 месяца). Средняя частота посещений за всё время составляет 1.72 раза в неделю, а за последний месяц - 1.55 раза. Средние траты на дополнительные услуги также ниже, всего 137. Доля клиентов, пришедших по промо друзей, низкая и равна 18%, а доля сотрудников компаний-партнеров равна 34%.

**В кластере 1 сосредоточились лояльные, активные клиенты.**

Доля ушедших здесь всего 3% и эти клиенты имеют длинный абонемент (Contract_period = 9.81 месяца), больший стаж (Lifetime = 4.80 месяца) и абонемент, который заканчивается не скоро (Month_to_end_contract = 8.96 месяца). Они посещают клуб в среднем 2.15 раза в неделю за всё время и 2.14 раза за последний месяц. Средние траты на дополнительные услуги выше чем в первой группе и составляют около 163. Значительная часть этих клиентов - сотрудники компаний-партнеров (около 74%), а около 53% пришли по промо друзей, а также живут рядом с клубом 94%.


## 8. Вычисление метрик качества кластеризации

Для объективной оценки того, насколько хорошо алгоритм разделил данные, необходимо рассчитать метрики качества.
Необходимо оценить результат кластеризации по внутренним и внешним метрикам и сравнить результат кластеризации с переменной Churn.

Для этого нужно рассчитать четыре метрики с помощью модуля sklearn.metrics:
* **Коэффициент силуэта (Silhouette Score)** - внутренняя метрика. Показывает, насколько объект похож на свой собственный кластер по сравнению с другими кластерами. Значения лежат в диапазоне от -1 до 1. Чем ближе значение к 1, тем лучше разделены кластеры. Значение около 0 указывает на перекрывающиеся кластеры;
* **Индекс Калински-Харабаса (Calinski-Harabasz Index)** - внутренняя метрика. Отношение межкластерной дисперсии к внутрикластерной. Чем выше значение, тем лучше;
* **Индекс Дэвиса-Болдина (Davies-Bouldin Index)** - внутренняя метрика. Оценивает среднее "сходство" между кластерами. Чем ближе значение к 0, тем лучше.
* **Скорректированный индекс Рэнда (Adjusted Rand Index, ARI)** - внешняя метрика. Сравнивает полученное разбиение с истинным (Churn). Значения от -1 до 1, где 1 означает полное совпадение, 0 - случайное совпадение.

In [ ]:
from sklearn.metrics import (calinski_harabasz_score,
                             davies_bouldin_score,
                             adjusted_rand_score)

# Вычисление внутренних метрик на стандартизированных данных
sil_score = silhouette_score(X_scaled, clusters)
ch_score = calinski_harabasz_score(X_scaled, clusters)
db_score = davies_bouldin_score(X_scaled, clusters)

# Вычисление внешней метрики (сравнение с истинной переменной Churn)
ari_score = adjusted_rand_score(target, clusters)

print("Метрики качества кластеризации")
print(f"1. Silhouette score:          {sil_score:.4f} (чем ближе к 1, тем лучше)")
print(f"2. Calinski-Harabasz:   {ch_score:.2f} (чем выше -  лучше)")
print(f"3. Davies-Bouldin:      {db_score:.4f} (чем ближе к 0, тем лучше)")
print(f"4. Adjusted Rand index (ARI): {ari_score:.4f} (от -1 до 1, чем выше, тем лучше)")

Анализ выполнения кода показал следующие результаты:

1. Silhouette score = 0.1647. Значение положительное, но невысокое. Это говорит о том, что алгоритм всё же нашёл некоторую структуру в данных - объекты в среднем ближе к центроидам своих кластеров, чем к центроидам соседних, но низкое значение указывает на то, что границы между кластерами размыты и кластеры частично перекрываются.
2. Calinski-Harabasz = 729.59. Высокое положительное значение, которое может свидетельствовать о наличии выраженной структуры разделения данных.
3. Davies-Bouldin = 2.2124. Значение заметно выше нуля, что подтверждает вывод силуэта о перекрытии кластеров. Внутрикластерное расстояние сопоставимо с межкластерным, то есть группы не являются сильно изолированными.
4. Adjusted Rand Index (ARI) = 0.0302. Значение близко к нулю, что означает практически случайное совпадение найденных кластеров с истинным разделением по признаку Churn. Алгоритм "K-Means" выделил две группы клиентов, но эти группы слабо коррелируют с фактом ухода из клуба.

Метрики подтверждают, что алгоритм "K-Means" нашёл в данных некоторую структуру, но эта она слабо связана с целевой переменной Churn.


## 9. Иерархическая агломеративная кластеризация

Для данного вида кластеризации сначала необходимо построить дендрограмму для визуального определения оптимального числа кластеров.

В отличие от "K-Means", этот алгоритм не требует заранее задавать количество кластеров. Он начинает работу с того, что каждый объект считается отдельным кластером, и на каждом шаге последовательно объединяет две наиболее похожие группы, пока не останется заданное число кластеров.

После оценки дендограммы обучить модель AgglomerativeClustering, рассчитать средние показатели по кластерам с помощью метода groupby и вычислть те же метрики качества, что и для K-Means, чтобы сравнить результаты.

Для построения дендрограммы использованы функции linkage() и dendrogram() из модуля scipy.cluster.hierarchy с методом Уорда (ward), который минимизирует дисперсию объединяемых кластеров



In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

# Вычисляем матрицу связей с использованием метода Уорда
linked = linkage(X_scaled, method='ward')

# Построение дендрограммы
plt.figure(figsize=(14, 6))
dendrogram(linked,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=False,
           truncate_mode='level',
           p=5,
           color_threshold=60)

plt.title('Дендрограмма иерархической кластеризации (метод Уорда)')
plt.xlabel('Образцы (клиенты)')
plt.ylabel('Евклидово расстояние')
plt.grid(True, alpha=0.3)
plt.show()

Анализ дендрограммы показывает иерархическую структуру данных.

Самое крупное объединение всех объектов в один кластер происходит на уровне расстояния около 105. Если провести горизонтальную линию на уровне расстояния около 90, данные разделятся на две крупные группы, что подтверждает обоснованность выбора k=2.
Также видно, что на уровне расстояния около 65 левая группа делится ещё на две подгруппы, что позволило бы выделить 3 кластера. На уровне около 35 можно было бы выделить 5 кластеров. Но резкий скачок именно на уровне 90 указывает на то, что предпочтительнее разделение данных на две группы.

Далее необходимо обучить модель AgglomerativeClustering из модуля sklearn.cluster с k=2, вычислить средние показатели по каждому кластеру с помощью groupby и сравнить метрики качества с результатами k-means.

In [ ]:

# Обучение агломеративной модели с k=2
agglo = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='ward')
clusters_agglo = agglo.fit_predict(X_scaled)

# Добавляем метки в датафрейм для анализа
df_analysis = df.copy()
df_analysis['Cluster_Agglo'] = clusters_agglo

print("Средние показатели по кластерам (Агломеративная кластеризация):")
display(df_analysis.groupby('Cluster_Agglo').mean().round(2))

# Вычисление метрик качества
sil_score_agglo = silhouette_score(X_scaled, clusters_agglo)
ch_score_agglo = calinski_harabasz_score(X_scaled, clusters_agglo)
db_score_agglo = davies_bouldin_score(X_scaled, clusters_agglo)
ari_score_agglo = adjusted_rand_score(target, clusters_agglo)

# Сравнительная таблица метрик
comparison_df = pd.DataFrame({
    'Метрика': ['Silhouette score', 'Calinski-Harabasz', 'Davies-Bouldin', 'Adjusted Rand Index'],
    'K-Means': [sil_score, ch_score, db_score, ari_score],
    'Agglomerative': [sil_score_agglo, ch_score_agglo, db_score_agglo, ari_score_agglo]
})

print("\nСравнение метрик качества:")
display(comparison_df)

Из анализа средних показателей по кластерам, полученным методом "Агломеративной кластеризации", видно схожую с "K-Means" картину разделения клиентов.

Кластер 0 объединяет клиентов с высоким риском ухода. Доля ушедших составляет 34%, средняя длительность абонемента всего 2.75 месяца, время с момента первого обращения равна 3.55 месяца, а частота посещений ниже.

Кластер 1 состоит из лояльных клиентов среди которых отток всего 4%, абонемент на 10.73 месяца, стаж больше, посещают зал чаще.

Получается, что оба метода выделили одинаковые по смыслу группы.

Сравнение метрик качества показывает, что метод K-Means справился с задачей лучше. По всем четырем метрикам K-Means показал более высокие результаты:
1. Silhouette Score выше на 0.05.
2. Calinski-Harabasz Index выше на 259.
3. Davies-Bouldin Index ниже на 0.14.
4. Adjusted Rand Index для агломеративной кластеризации оказался отрицательным и равен -0.06. Отрицательное значение ARI означает, что разбиение, полученное агломеративным методом, согласуется с истинным распределением по признаку Churn даже хуже, чем случайное.

По итогу иерархическая кластеризация подтвердила наличие в данных двух основных групп, но по количественным метрикам уступила методу "K-Means".


## 10. Выводы по проделанной работе

В ходе выполнения лабораторной работы я изучили и применили на практике алгоритмы кластеризации для анализа клиентской базы фитнес-клуба.

Мной были пройдены все ключевые этапы работы с данными: от первичного осмотра и визуализации до стандартизации, подбора оптимального числа кластеров и оценки качества разбиения.

На основании проведенного анализа я выяснил, что клиенты фитнес-клуба успешно разделяются на две смысловые группы:
1. Первая группа объединяет лояльных клиентов которые имеют длительный стаж, высокий абонемент, регулярно посещают клуб и демонстрируют минимальный отток. 2. Вторая группа состоит из клиентов с высоким риском ухода которые посещают зал редко, имеют короткий абонемент и чаще всего покидают клуб.

Такое разделение имеет высокую практическую ценность для бизнеса, так как позволяет маркетологам применять точечные стратегии удержания именно для второй группы.

Сравнение алгоритма "K-Means" и "Иерархической агломеративной кластеризации" показало, что метод "K-Means" дал более высокие значения метрик качества, а "Агломеративный метод" оказался отличным инструментом для анализа, но итоговые метрики самого разбиения у него оказались ниже.

Из проделанной работы можно сделать вывод о том, что для решения подобных задач сегментации метод "K-Means" является более предпочтительным. Он работает значительно быстрее, требует меньше вычислительных ресурсов, легче масштабируется на большие объемы данных и показал лучшие результаты по внутренним и внешним метрикам. Иерархическая кластеризация отлично подходит для визуального анализа структуры данных и выбора параметра k, но имеет высокую вычислительную сложность.
